In [1]:
import requests
from bs4 import BeautifulSoup
import json, time, random, re
import pandas as pd

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [3]:
annonces = []

villes = [
    ("paris", "g439"),
    ("lyon", "g69"),
    ("bordeaux", "g33"),
    ("marseille", "g13"),
    ("toulouse", "g31"),
    ("nantes", "g44"),
    ("nice", "g06"),
    ("lille", "g59"),
    ("montpellier", "g34"),
    ("rennes", "g35"),
]

types = ["ventes-appartements", "locations-appartements", "ventes-maisons", "locations-maisons"]

for type_bien in types:
    for ville_nom, ville_code in villes:
        for page in range(1, 16):
            url = f"https://www.pap.fr/annonce/{type_bien}-{ville_nom}-{ville_code}/p-{page}"
            try:
                r = requests.get(url, headers=headers, timeout=10)
                soup = BeautifulSoup(r.text, "html.parser")

                # Utiliser item-body comme conteneur principal
                cards = soup.select("div.item-body")
                if not cards:
                    print(f"Fin pour {type_bien}/{ville_nom} page {page}")
                    break

                for card in cards:
                    # Prix
                    prix_tag = card.select_one("span.item-price")
                    prix_text = prix_tag.get_text(strip=True) if prix_tag else ""
                    prix_match = re.search(r'([\d\s\.]+)', prix_text.replace('\xa0', ' '))
                    prix = prix_match.group(1).replace(" ", "").replace(".", "") if prix_match else ""

                    # Titre/localisation
                    titre_tag = card.select_one("a.item-title span.h1")
                    titre = titre_tag.get_text(strip=True) if titre_tag else ""

                    # Tags (surface, pièces)
                    tags = card.select("ul.item-tags li")
                    tags_text = [t.get_text(strip=True) for t in tags]

                    surface_match = next((re.search(r'([\d,]+)\s*m²', t) for t in tags_text if 'm²' in t), None)
                    surface = surface_match.group(1).replace(",", ".") if surface_match else ""

                    pieces_match = next((re.search(r'(\d+)\s*pièce', t) for t in tags_text if 'pièce' in t), None)
                    pieces = pieces_match.group(1) if pieces_match else ""

                    chambres_match = next((re.search(r'(\d+)\s*chambre', t) for t in tags_text if 'chambre' in t), None)
                    chambres = chambres_match.group(1) if chambres_match else ""

                    # Description
                    desc_tag = card.select_one("p.item-description")
                    description = desc_tag.get_text(strip=True) if desc_tag else ""

                    # On vérifie si le titre contient un code postal français (5 chiffres)
                    # Si ce n'est pas le cas, on passe à l'annonce suivante
                    cp_match = re.search(r'\((\d{5})\)', titre)
                    if not cp_match:
                        continue 
                    
                    # Optionnel : tu peux aussi créer une colonne CP propre
                    code_postal = cp_match.group(1)

                    if prix:
                        annonces.append({
                            "type": type_bien,
                            "ville": titre if titre else ville_nom,
                            "prix": prix,
                            "surface": surface,
                            "nb_pieces": pieces,
                            "nb_chambres": chambres,
                            "description": description,
                        })

                print(f"[{type_bien} - {ville_nom} - p.{page}] → {len(annonces)} annonces total")
                time.sleep(random.uniform(3, 7))

            except Exception as e:
                print(f"Erreur : {e}")

# Sauvegarde
df = pd.DataFrame(annonces)

[ventes-appartements - paris - p.1] → 9 annonces total
[ventes-appartements - paris - p.2] → 18 annonces total
[ventes-appartements - paris - p.3] → 29 annonces total
[ventes-appartements - paris - p.4] → 38 annonces total
[ventes-appartements - paris - p.5] → 49 annonces total
[ventes-appartements - paris - p.6] → 60 annonces total
[ventes-appartements - paris - p.7] → 72 annonces total
[ventes-appartements - paris - p.8] → 84 annonces total
[ventes-appartements - paris - p.9] → 93 annonces total
[ventes-appartements - paris - p.10] → 103 annonces total
[ventes-appartements - paris - p.11] → 114 annonces total
[ventes-appartements - paris - p.12] → 126 annonces total
[ventes-appartements - paris - p.13] → 138 annonces total
[ventes-appartements - paris - p.14] → 152 annonces total
[ventes-appartements - paris - p.15] → 167 annonces total
[ventes-appartements - lyon - p.1] → 167 annonces total
[ventes-appartements - lyon - p.2] → 167 annonces total
[ventes-appartements - lyon - p.3] → 

In [4]:
# Nettoyage rapide pour éviter les sauts de ligne dans les colonnes
df = df.replace(r'\n', ' ', regex=True).replace(r'\r', ' ', regex=True)
print(f"{len(annonces)} annonces")
print(df.head())
print(df.isnull().sum())

1470 annonces
                  type              ville    prix surface nb_pieces  \
0  ventes-appartements  Paris 10E (75010)  610000      65         3   
1  ventes-appartements  Paris 13E (75013)  532000      65         3   
2  ventes-appartements   Paris 5E (75005)  128000    9.31         1   
3  ventes-appartements  Paris 18E (75018)  330000      28         2   
4  ventes-appartements  Paris 19E (75019)  850000      72         4   

  nb_chambres                                        description  
0           2  3 pièces de 65 m², au 154 rue La Fayette  au c...  
1           2  Paris 13 ème – Quartier Jeanne d'Arc.  Dans un...  
2           1  Chambre Métro Maubert-Mutualité.  Au 6ᵉ étage ...  
3           1  Appartement 2 pièces – Paris 18e – 28 m² – 335...  
4           2  À l'abri de l'agitation urbaine, ce superbe du...  
type           0
ville          0
prix           0
surface        0
nb_pieces      0
nb_chambres    0
description    0
dtype: int64


In [7]:
# Sauvegarde du fichier 
df.to_csv("pap_raw.csv", 
          sep=";", 
          index=False, 
          encoding="utf-8-sig", 
          quoting=1) # Ajoute des guillemets pour protéger les textes longs
with open("pap_raw.json", "w", encoding="utf-8") as f:
    json.dump(annonces, f, ensure_ascii=False, indent=2)